# SkillPulse — Phase 2: Data Cleaning & NLP Skill Extraction

This notebook loads the raw job postings from the MySQL database, cleans the text, removes duplicate postings, extracts key technical skills using optimized regular expressions, and populates the `skills` and `job_skills` tables to establish our relational data structures.

## 1. Imports and Environment Setup

In [1]:
import os
import re
import hashlib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, Table, Column, Integer, String, MetaData
from dotenv import load_dotenv
from urllib.parse import quote_plus

# Load environment variables
load_dotenv(dotenv_path="../.env")

True

## 2. Database Connection

In [2]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# Establish SQLAlchemy engine, encoding the password to support special characters like '@'
engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))
    print("Connected successfully! Tables found:", [row[0] for row in result])

Connected successfully! Tables found: ['drift_reports', 'job_postings', 'job_skills', 'model_runs', 'skills']


## 3. Load Raw Data

In [3]:
# Load raw job postings
query = "SELECT id, source, country, title, company, location, salary_min, salary_max, description FROM job_postings"
df_raw = pd.read_sql(query, con=engine)
print(f"Loaded {len(df_raw)} raw job postings.")
df_raw.head()

Loaded 12100 raw job postings.


,id,source,country,title,company,location,salary_min,salary_max,description
0,4051,adzuna,in,Data Science Instructor,NDMIT,"Agra, Uttar Pradesh",NaN,NaN,Data Science Instructor: NDMIT is seeking a pa...
1,4052,adzuna,in,Data Science Instructor,upGrad,India,NaN,NaN,Data Science & AI Instructor upGrad is expandi...
2,4053,adzuna,in,Principal Data Scientist,Warner Bros. Discovery,"Hyderabad, Telangana",NaN,NaN,"This job is with Warner Bros. Discovery, an in..."
3,4054,adzuna,in,Data Scientist III,S&P Global,"Hyderabad, Telangana",NaN,NaN,"This job is with S&P Global, an inclusive empl..."
4,4055,adzuna,in,Data Science Instructor,upGrad,India,NaN,NaN,Job Title: Data Science Instructor – Offline L...


## 4. String Cleansing & Normalization

We need to clean the text data, especially `description` and `title` which contain HTML formatting fragments like `<strong>`, `<br>`, and entities like `&amp;`.

In [4]:
def clean_html_text(text_val):
    if not text_val or not isinstance(text_val, str):
        return ""
    # Remove HTML tags using regex
    clean = re.sub(r'<[^>]+>', ' ', text_val)
    # Clean up standard HTML entities
    clean = clean.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"').replace('&#39;', "'")
    # Replace multiple spaces/newlines with a single space
    clean = re.sub(r'\s+', ' ', clean)
    return clean.strip()

# Create copy and apply cleansing
df_cleaned = df_raw.copy()
df_cleaned['title_clean'] = df_cleaned['title'].apply(clean_html_text)
df_cleaned['company_clean'] = df_cleaned['company'].apply(clean_html_text)
df_cleaned['description_clean'] = df_cleaned['description'].apply(clean_html_text)

print("Sample description before cleaning:")
print(df_raw['description'].iloc[0][:300])
print("\nSample description after cleaning:")
print(df_cleaned['description_clean'].iloc[0][:300])

Sample description before cleaning:
Data Science Instructor: NDMIT is seeking a passionate and knowledgeable Data Science Instructor to join our Kanpur branch. The ideal candidate should have hands-on experience in Data Science concepts and the ability to train students through practical and industry-oriented sessions. Key Responsibil

Sample description after cleaning:
Data Science Instructor: NDMIT is seeking a passionate and knowledgeable Data Science Instructor to join our Kanpur branch. The ideal candidate should have hands-on experience in Data Science concepts and the ability to train students through practical and industry-oriented sessions. Key Responsibil


## 5. Deduplication

Duplicate job postings can significantly skew our modeling and trend tracking. We will deduplicate based on a composite key composed of:
* Cleaned job title
* Cleaned company name
* Target country
* The first 200 characters of the cleaned description (to handle small variations in posting timestamps or locations while identifying identical job specs).

In [5]:
def generate_dedup_hash(row):
    # Normalize inputs
    t = str(row['title_clean']).lower().strip()
    c = str(row['company_clean']).lower().strip()
    co = str(row['country']).lower().strip()
    # Use first 200 chars of description as part of fingerprint
    d = str(row['description_clean']).lower().strip()[:200]
    
    composite = f"{t}|||{c}|||{co}|||{d}"
    return hashlib.md5(composite.encode('utf-8')).hexdigest()

df_cleaned['dedup_hash'] = df_cleaned.apply(generate_dedup_hash, axis=1)

original_count = len(df_cleaned)
# Drop duplicates keeping the first occurrence
df_unique = df_cleaned.drop_duplicates(subset=['dedup_hash'], keep='first')
unique_count = len(df_unique)

print(f"Original row count: {original_count}")
print(f"Deduplicated row count: {unique_count}")
print(f"Removed {original_count - unique_count} duplicate rows ({((original_count - unique_count)/original_count)*100:.2f}% of dataset).")

Original row count: 12100
Deduplicated row count: 4558
Removed 7542 duplicate rows (62.33% of dataset).


## 6. Define Skill Mapping Dictionary

Here we define our dictionary of developer skills. Each skill is mapped to a compiled regex pattern using raw strings and word boundary `\b` anchors to prevent matching substrings (like matching 'Go' in 'Google' or 'Java' in 'JavaScript').

In [6]:
SKILLS_PATTERNS = {
    # Languages
    'Python': re.compile(r'\b(python|py)\b', re.IGNORECASE),
    'SQL': re.compile(r'\b(sql|mysql|postgresql|sqlite|oracle|mssql|plsql)\b', re.IGNORECASE),
    'JavaScript': re.compile(r'\b(javascript|js|es6)\b', re.IGNORECASE),
    'TypeScript': re.compile(r'\b(typescript|ts)\b', re.IGNORECASE),
    'Java': re.compile(r'\b(java)\b', re.IGNORECASE), # note: Javascript is handled by boundary or matches separately
    'C++': re.compile(r'\bc\+\+\b', re.IGNORECASE),
    'C#': re.compile(r'\bc#|c-sharp\b', re.IGNORECASE),
    'Go': re.compile(r'\b(golang|go\s*lang)\b|\bGo\b', re.ASCII),  # ASCII is safer, we will match case-sensitive or with contextual regex
    'Ruby': re.compile(r'\b(ruby|rails)\b', re.IGNORECASE),
    'Rust': re.compile(r'\b(rust)\b', re.IGNORECASE),
    'HTML/CSS': re.compile(r'\b(html5?|css3?|sass|scss|less)\b', re.IGNORECASE),
    'PHP': re.compile(r'\b(php)\b', re.IGNORECASE),
    'Kotlin': re.compile(r'\b(kotlin)\b', re.IGNORECASE),
    'Swift': re.compile(r'\b(swift)\b', re.IGNORECASE),
    
    # Frameworks
    'React': re.compile(r'\b(react|reactjs|react\.js)\b', re.IGNORECASE),
    'Angular': re.compile(r'\b(angular|angularjs|angular\.js)\b', re.IGNORECASE),
    'Vue': re.compile(r'\b(vue|vuejs|vue\.js)\b', re.IGNORECASE),
    'FastAPI': re.compile(r'\b(fastapi|fast-api)\b', re.IGNORECASE),
    'Flask': re.compile(r'\b(flask)\b', re.IGNORECASE),
    'Django': re.compile(r'\b(django)\b', re.IGNORECASE),
    'Node.js': re.compile(r'\b(node|nodejs|node\.js)\b', re.IGNORECASE),
    'Spring Boot': re.compile(r'\b(spring\s*boot|spring\s*mvc|spring)\b', re.IGNORECASE),
    'Next.js': re.compile(r'\b(nextjs|next\.js)\b', re.IGNORECASE),
    
    # Databases
    'MongoDB': re.compile(r'\b(mongo|mongodb)\b', re.IGNORECASE),
    'Redis': re.compile(r'\b(redis)\b', re.IGNORECASE),
    'Cassandra': re.compile(r'\b(cassandra)\b', re.IGNORECASE),
    'Elasticsearch': re.compile(r'\b(elasticsearch|elastic)\b', re.IGNORECASE),
    'DynamoDB': re.compile(r'\b(dynamodb)\b', re.IGNORECASE),
    
    # Tools & Platforms
    'AWS': re.compile(r'\b(aws|amazon\s*web\s*services|ec2|s3)\b', re.IGNORECASE),
    'Azure': re.compile(r'\b(azure|microsoft\s*azure)\b', re.IGNORECASE),
    'GCP': re.compile(r'\b(gcp|google\s*cloud|google\s*cloud\s*platform)\b', re.IGNORECASE),
    'Docker': re.compile(r'\b(docker|containers)\b', re.IGNORECASE),
    'Kubernetes': re.compile(r'\b(kubernetes|k8s)\b', re.IGNORECASE),
    'Git': re.compile(r'\b(git|github|gitlab)\b', re.IGNORECASE),
    'Jenkins': re.compile(r'\b(jenkins)\b', re.IGNORECASE),
    'Terraform': re.compile(r'\b(terraform)\b', re.IGNORECASE),
    'CI/CD': re.compile(r'\b(ci/cd|cicd|continuous\s*integration|continuous\s*deployment)\b', re.IGNORECASE),
    'Airflow': re.compile(r'\b(airflow|apache\s*airflow)\b', re.IGNORECASE),
    'Snowflake': re.compile(r'\b(snowflake)\b', re.IGNORECASE),
    'Kafka': re.compile(r'\b(kafka|apache\s*kafka)\b', re.IGNORECASE),
    
    # Concepts
    'Agile': re.compile(r'\b(agile|scrum|kanban)\b', re.IGNORECASE),
    'Machine Learning': re.compile(r'\b(machine\s*learning|ml)\b', re.IGNORECASE),
    'Deep Learning': re.compile(r'\b(deep\s*learning|dl)\b', re.IGNORECASE),
    'NLP': re.compile(r'\b(nlp|natural\s*language\s*processing)\b', re.IGNORECASE),
    'DevOps': re.compile(r'\b(devops)\b', re.IGNORECASE),
    'Microservices': re.compile(r'\b(microservices|micro-services)\b', re.IGNORECASE),
    'API': re.compile(r'\b(api|apis|restful|rest\s*api)\b', re.IGNORECASE)
}

# Go matching needs customized logic to avoid case-insensitive matching of the verb "go"
def extract_skills_from_text(title, description):
    combined_text = f"{title} {description}"
    matched = []
    
    for skill, pattern in SKILLS_PATTERNS.items():
        if skill == 'Go':
            # Special case-sensitive matching for "Go"
            # Match "golang", "go lang" case-insensitively, or capital "Go" with word boundaries
            if re.search(r'\b(golang|go\s*lang)\b', combined_text, re.IGNORECASE) or re.search(r'\bGo\b', combined_text):
                matched.append(skill)
        elif skill == 'Java':
            # Special check to make sure Java isn't matching JavaScript
            # (Though Javascript regex is \b(javascript|js|es6)\b, java \b(java)\b can match 'java' in 'java developer')
            # We use word boundary so it won't match 'javascript'
            if pattern.search(combined_text):
                matched.append(skill)
        else:
            if pattern.search(combined_text):
                matched.append(skill)
                
    return list(set(matched))

## 7. Run Skill Extraction

Apply the skill extraction to each unique job posting, creating a mapping of job ID to matched skills.

In [7]:
print("Extracting skills from job postings...")
df_unique = df_unique.copy()
df_unique['matched_skills'] = df_unique.apply(
    lambda row: extract_skills_from_text(row['title_clean'], row['description_clean']),
    axis=1
)

df_unique['skills_count'] = df_unique['matched_skills'].apply(len)
print("Skill extraction complete!")
print(f"Jobs with at least one skill: {sum(df_unique['skills_count'] > 0)} out of {len(df_unique)} ({sum(df_unique['skills_count'] > 0)/len(df_unique)*100:.2f}%)")

Extracting skills from job postings...
Skill extraction complete!
Jobs with at least one skill: 2570 out of 4558 (56.38%)


## 8. Populate the Relational Tables (`skills` and `job_skills`)

Now we write the skills and mappings back to the database. We will make sure that the `skills` table is populated with all our defined canonical skills, retrieve their IDs, and then bulk-insert mappings into `job_skills`.

In [8]:
from sqlalchemy.dialects.mysql import insert as mysql_insert

# 1. Populate 'skills' table
all_canonical_skills = list(SKILLS_PATTERNS.keys())
skills_data = [{"name": s} for s in all_canonical_skills]

with engine.begin() as conn:
    # Insert skills, ignoring duplicates
    metadata = MetaData()
    skills_table = Table('skills', metadata, autoload_with=engine)
    
    for skill_dict in skills_data:
        # Use MySQL-specific INSERT ... ON DUPLICATE KEY UPDATE to avoid errors
        stmt = mysql_insert(skills_table).values(name=skill_dict['name'])
        on_dup_stmt = stmt.on_duplicate_key_update(name=stmt.inserted.name)
        conn.execute(on_dup_stmt)
        
    print("Skills master table populated.")

# 2. Load the map of skill_name -> skill_id from database
with engine.connect() as conn:
    df_skills_db = pd.read_sql("SELECT id, name FROM skills", con=conn)
skill_to_id = dict(zip(df_skills_db['name'], df_skills_db['id']))

# 3. Prepare mappings for 'job_skills'
job_skills_rows = []
for _, row in df_unique.iterrows():
    job_id = int(row['id'])
    for skill_name in row['matched_skills']:
        skill_id = skill_to_id.get(skill_name)
        if skill_id is not None:
            job_skills_rows.append({
                "job_id": job_id,
                "skill_id": skill_id
            })

print(f"Prepared {len(job_skills_rows)} job-skill relationships.")

# 4. Bulk insert into 'job_skills'
# Note: To avoid duplicate primary key errors (job_id, skill_id) if run multiple times, 
# we can clear the table or use ON DUPLICATE KEY UPDATE.
with engine.begin() as conn:
    # Clear existing job_skills to prevent duplicate entries
    conn.execute(text("DELETE FROM job_skills"))
    print("Cleared existing entries in job_skills.")
    
    if job_skills_rows:
        # Bulk insert using SQLAlchemy Core
        job_skills_table = Table('job_skills', metadata, autoload_with=engine)
        conn.execute(job_skills_table.insert(), job_skills_rows)
        print(f"Inserted {len(job_skills_rows)} relationships into job_skills.")

Skills master table populated.
Prepared 5548 job-skill relationships.
Cleared existing entries in job_skills.
Inserted 5548 relationships into job_skills.


## 9. Data Quality & Insights Reporting

In [9]:
# Flatten matching results for aggregate statistics
all_matched_flat = [s for lst in df_unique['matched_skills'] for s in lst]
df_stats = pd.Series(all_matched_flat).value_counts().reset_index()
df_stats.columns = ['Skill', 'Count']
df_stats['Percentage'] = (df_stats['Count'] / len(df_unique)) * 100

print("\n--- Top 15 Most Demanded Skills ---")
print(df_stats.head(15))

# Check skill distribution by country
country_skill_data = []
for _, row in df_unique.iterrows():
    c = row['country']
    for s in row['matched_skills']:
        country_skill_data.append({'country': c, 'skill': s})

if country_skill_data:
    df_country_skill = pd.DataFrame(country_skill_data)
    df_pivot = df_country_skill.groupby(['country', 'skill']).size().unstack(fill_value=0)
    # Get top 5 skills per country
    for c in df_pivot.index:
        print(f"\nTop 5 skills in {c.upper()}:")
        print(df_pivot.loc[c].sort_values(ascending=False).head(5))
else:
    print("No country skill data available.")


--- Top 15 Most Demanded Skills ---
               Skill  Count  Percentage
0   Machine Learning   1321   28.982010
1             Python    445    9.763054
2               Java    396    8.688021
3                API    393    8.622203
4                SQL    285    6.252742
5         JavaScript    239    5.243528
6                AWS    204    4.475647
7            Node.js    175    3.839403
8      Microservices    163    3.576130
9              Agile    141    3.093462
10       Spring Boot    135    2.961825
11               NLP    127    2.786310
12     Deep Learning    118    2.588855
13                C#    118    2.588855
14             Azure    110    2.413339

Top 5 skills in GB:
skill
Machine Learning    432
Python              118
Java                115
API                  97
C#                   76
Name: gb, dtype: int64

Top 5 skills in IN:
skill
Machine Learning    664
Python              257
API                 177
SQL                 163
JavaScript          150
Name: 